In [1]:
import os, time, math
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from empath import Empath
from transformers import DistilBertTokenizer, DistilBertModel, get_linear_schedule_with_warmup

# --- CONFIGURATION ---
TEXT_COL = 'text'
LABEL_COL = 'real'
MAX_LEN = 80 # Reduced from 128 for speed
BATCH_SIZE = 32
EPOCHS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 1. DATA LOADING & EMPATH FEATURES ---
df = pd.read_csv("data_cleaned.csv")
lexicon = Empath()

# Feature extraction: Empath Lexicon
REQUESTED_CATS = ["joy", "positive_emotion", "sadness", "anger", "negative_emotion", 
                  "social", "achievement", "affection", "violence", "disgust", "fear"]

def get_empath_features(texts):
    print("Extracting Empath features...")
    # Empath is CPU-bound; this is often the bottleneck before training starts
    features = []
    for text in texts:
        score = lexicon.analyze(str(text), categories=REQUESTED_CATS, normalize=True)
        if score is None: score = {c: 0.0 for c in REQUESTED_CATS}
        features.append([score.get(c, 0.0) for c in REQUESTED_CATS])
    return np.array(features)

E_features = get_empath_features(df[TEXT_COL])
scaler = StandardScaler()
E_scaled = scaler.fit_transform(E_features)

# --- 2. TRAIN/TEST SPLIT ---
X_train_text, X_test_text, E_train, E_test, y_train, y_test = train_test_split(
    df[TEXT_COL].values, E_scaled, df[LABEL_COL].values, test_size=0.2, random_state=42
)

# --- 3. PYTORCH DATASET ---
tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

class FakeNewsDataset(Dataset):
    def __init__(self, texts, empath, labels):
        self.texts = texts
        self.empath = empath
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = tokenizer(text, truncation=True, padding='max_length', 
                             max_length=MAX_LEN, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'empath': torch.tensor(self.empath[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_loader = DataLoader(FakeNewsDataset(X_train_text, E_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(FakeNewsDataset(X_test_text, E_test, y_test), batch_size=BATCH_SIZE)

# --- 4. OPTIMIZED HYBRID MODEL ---
class HybridModel(nn.Module):
    def __init__(self, num_classes, empath_dim):
        super(HybridModel, self). __init__()
        # DistilBert is much faster than standard BERT
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        
        # FREEZE BERT: This makes training 10x faster
        for param in self.distilbert.parameters():
            param.requires_grad = False
            
        self.lstm = nn.LSTM(768, 128, batch_first=True, bidirectional=True)
        self.gru = nn.GRU(256, 64, batch_first=True, bidirectional=True)
        
        self.empath_dense = nn.Linear(empath_dim, 32)
        self.classifier = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )

    def forward(self, input_ids, attention_mask, empath):
        # Bert output
        bert_out = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = bert_out.last_hidden_state # [batch, seq, 768]
        
        # Recurrent Layers
        lstm_out, _ = self.lstm(sequence_output)
        gru_out, _ = self.gru(lstm_out)
        avg_pool = torch.mean(gru_out, 1)
        
        # Empath side
        empath_out = torch.relu(self.empath_dense(empath))
        
        # Concatenate
        combined = torch.cat((avg_pool, empath_out), dim=1)
        return self.classifier(combined)

model = HybridModel(num_classes=2, empath_dim=len(REQUESTED_CATS)).to(DEVICE)

# --- 5. TRAINING WITH EARLY STOPPING ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # Slightly higher LR since BERT is frozen
criterion = nn.CrossEntropyLoss()

best_accuracy = 0
patience = 2
trigger_times = 0

print(f"Starting training on {DEVICE}...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        empath = batch['empath'].to(DEVICE)
        labels = batch['label'].to(DEVICE)
        
        outputs = model(input_ids, mask, empath)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            outputs = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['empath'].to(DEVICE))
            _, predicted = torch.max(outputs.data, 1)
            total += batch['label'].size(0)
            correct += (predicted == batch['label'].to(DEVICE)).sum().item()
    
    accuracy = correct / total
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {accuracy:.4f}")
    
    # Early Stopping Logic
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        trigger_times = 0
        torch.save(model.state_dict(), 'best_hybrid_model.pt')
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping!")
            break

Extracting Empath features...


c:\Users\agmil\anaconda3\envs\dev\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Starting training on cpu...
Epoch 1 | Loss: 0.0749 | Val Acc: 0.9982
Epoch 2 | Loss: 0.0076 | Val Acc: 0.9988
Epoch 3 | Loss: 0.0043 | Val Acc: 0.9992
Epoch 4 | Loss: 0.0037 | Val Acc: 0.9992
Epoch 5 | Loss: 0.0040 | Val Acc: 0.9991
Early stopping!
